In [1]:
from preprocessing_class import Preprocessing, load_movies, load_pres
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC,SVC
from nltk.corpus import stopwords
import numpy as np
import string
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


# Movies dataset

In [2]:
# load dataset
alltxt, alllabs = load_movies("./Dataset/movies1000/")
alltxt, alllabs = np.array(alltxt), np.array(alllabs)

len(alltxt), len(alllabs), np.unique(alllabs)

(2000, 2000, array([0, 1]))

In [3]:
# train test split here for test in final prediction

X_train, X_test, y_train, y_test = train_test_split(
    alltxt, alllabs, 
    test_size=0.20, 
    stratify=alllabs, 
    random_state=42
)

len(X_train), len(y_train), len(X_test), len(y_test)

(1600, 1600, 400, 400)

**Start with a CountVectorizer**

In [ ]:
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Initialize regex tokenizer
tokenizer = RegexpTokenizer(r'\w+')

# Vectorize document using TF-IDF
vectorizer = TfidfVectorizer(lowercase=True,
                        stop_words='english',
                        ngram_range = (1,1),
                        tokenizer = tokenizer.tokenize, max_df=0.95, min_df=2, max_features=1000)

vectors = vectorizer.fit_transform(X_train)
print(vectors.shape)
print(vectors.nnz / float(vectors.shape[0]))

/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


(1600, 1000)
111.86375


In [5]:
from sklearn.decomposition import LatentDirichletAllocation


lda = LatentDirichletAllocation(n_components=10)
M = lda.fit_transform(vectors)

In [6]:
M.shape # mots vs topics

(1600, 10)

In [7]:
topics = np.argmax(M, axis=1) # taking the max is a clutering way 
# can apply kmeans after lda, as we have a new representation dimension

In [8]:
kmeans = KMeans(n_clusters=10, random_state=42)
kmeans.fit(M)
labels = kmeans.labels_

In [9]:
# top words per topic
feat_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = feat_names[np.argsort(topic)[::-1][:10]]
    print(f"Topic {topic_idx}:", top_words)

Topic 0: ['film' 't' 'like' 'just' 'character' 'characters' 'good' 'movie' 'way'
 'story']
Topic 1: ['film' 'story' 'like' 'effects' 'films' 't' 'time' 'world' 'special'
 'original']
Topic 2: ['film' 'comedy' 'little' 'movie' '10' 'mr' 'best' 'school' 'max' 't']
Topic 3: ['action' 'film' 'jackie' 'films' 'like' 'director' 'plot' 't' 'time'
 'killer']
Topic 4: ['movie' 't' 'like' 'just' 'good' 'bad' 'film' 'funny' 'really' 'plot']
Topic 5: ['movie' 't' 'just' 'film' 'like' 'bad' 'know' 'joe' 'really' 'time']
Topic 6: ['film' 'war' 'movie' 'men' 'world' 'like' 'time' 'american' 'ryan'
 'great']
Topic 7: ['film' 'movie' 't' 'love' 'life' 'time' 'like' 'story' 'man' 'just']
Topic 8: ['disney' 'batman' 'film' 'robin' 'animated' 'van' 'vampire' 'animation'
 'comic' 'children']
Topic 9: ['wild' 'west' 'smith' 'american' 'movie' 'story' 'director' 'comedy'
 'white' 'black']


In [10]:
NN2cluster = np.argmax(np.abs(M), axis=0) 
#import seaborn as sns
import matplotlib.cm as cm
from matplotlib.pyplot import get_cmap
cmap = cm.tab20
cmap = get_cmap('hsv', 20)
cmap = get_cmap('jet', 20)
#cmap = cm.tab20

plt.figure(figsize=(15,10))
plt.scatter(tsne_mat[:,0],tsne_mat[:,1], c=Y, cmap=cmap, s=10,)
plt.scatter(tsne_mat[NN2cluster[:],0],tsne_mat[NN2cluster[:],1], c='black', s=100)
plt.colorbar(ticks=range(20))
# plt.legend()
#plt.scatter(tsne_mat[NN2cluster2[:],0],tsne_mat[NN2cluster2[:],1], c='red', s=100)

NameError: name 'tsne_mat' is not defined

<Figure size 1500x1000 with 0 Axes>